In [1]:
!pip install yfinance xgboost


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
from xgboost import XGBClassifier, XGBRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, mean_absolute_error

In [3]:
# Last 6 months of data
df = yf.download("AAPL", period="6mo", interval="1d")

if isinstance(df.columns, pd.MultiIndex):
    df.columns = [col[0] for col in df.columns]

df = df.reset_index()
df = df.sort_values("Date")

[*********************100%***********************]  1 of 1 completed


In [4]:
# --- Feature Engineering ---

df["sma_10"] = df["Close"].rolling(10).mean()
df["sma_20"] = df["Close"].rolling(20).mean()

delta = df["Close"].diff()
gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)
avg_gain = gain.rolling(14).mean()
avg_loss = loss.rolling(14).mean()
rs = avg_gain / avg_loss
df["rsi_14"] = 100 - (100 / (1 + rs))

ema12 = df["Close"].ewm(span=12, adjust=False).mean()
ema26 = df["Close"].ewm(span=26, adjust=False).mean()
df["macd"] = ema12 - ema26
df["macd_signal"] = df["macd"].ewm(span=9, adjust=False).mean()

rolling_mean = df["Close"].rolling(20).mean()
rolling_std = df["Close"].rolling(20).std()
df["bb_upper"] = rolling_mean + (2 * rolling_std)
df["bb_lower"] = rolling_mean - (2 * rolling_std)
df["bb_width"] = (df["bb_upper"] - df["bb_lower"]) / df["Close"]

df["return"] = df["Close"].pct_change()
df["return_3d"] = df["Close"].pct_change(3)
df["return_5d"] = df["Close"].pct_change(5)

df["volatility_5d"] = df["return"].rolling(5).std()
df["volatility_10d"] = df["return"].rolling(10).std()

df["dist_sma_10"] = (df["Close"] - df["sma_10"]) / df["sma_10"]
df["dist_sma_20"] = (df["Close"] - df["sma_20"]) / df["sma_20"]

# Targets
df["target_signal"] = (df["return"].shift(-1) > 0.002).astype(int)
df["target_price"]  = df["Close"].shift(-1)

df = df.dropna()

In [5]:
feature_cols = [
    "rsi_14", "macd", "macd_signal", "bb_width",
    "return_3d", "return_5d", "volatility_5d", "volatility_10d",
    "dist_sma_10", "dist_sma_20", "Volume"
]

X = df[feature_cols]
y_signal = df["target_signal"]
y_price  = df["target_price"]

split_index = int(len(df) * 0.8)

X_train, X_test         = X[:split_index], X[split_index:]
y_sig_train, y_sig_test = y_signal[:split_index], y_signal[split_index:]
y_prc_train, y_prc_test = y_price[:split_index], y_price[split_index:]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

In [6]:
# --- Classifier: BUY / SELL / HOLD signal ---
xgb_clf = XGBClassifier(
    n_estimators=500, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric="logloss", random_state=42
)
xgb_clf.fit(
    X_train_scaled, y_sig_train,
    eval_set=[(X_test_scaled, y_sig_test)],
    verbose=100
)

clf_pred = xgb_clf.predict(X_test_scaled)
print("XGB Classifier Accuracy:", accuracy_score(y_sig_test, clf_pred))
print(classification_report(y_sig_test, clf_pred))
print("ROC-AUC:", roc_auc_score(y_sig_test, xgb_clf.predict_proba(X_test_scaled)[:, 1]))

[0]	validation_0-logloss:0.67704
[100]	validation_0-logloss:0.93390
[200]	validation_0-logloss:1.04147
[300]	validation_0-logloss:1.11170
[400]	validation_0-logloss:1.20664
[499]	validation_0-logloss:1.28886
XGB Classifier Accuracy: 0.5238095238095238
              precision    recall  f1-score   support

           0       1.00      0.23      0.38        13
           1       0.44      1.00      0.62         8

    accuracy                           0.52        21
   macro avg       0.72      0.62      0.50        21
weighted avg       0.79      0.52      0.47        21

ROC-AUC: 0.5961538461538461


In [7]:
# --- Regressor: Next day closing price ---
xgb_reg = XGBRegressor(
    n_estimators=500, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, random_state=42
)
xgb_reg.fit(
    X_train_scaled, y_prc_train,
    eval_set=[(X_test_scaled, y_prc_test)],
    verbose=100
)

price_pred = xgb_reg.predict(X_test_scaled)
print("XGB Regressor MAE: $", round(mean_absolute_error(y_prc_test, price_pred), 2))

[0]	validation_0-rmse:6.94339
[100]	validation_0-rmse:7.87289
[200]	validation_0-rmse:8.02763
[300]	validation_0-rmse:8.01343
[400]	validation_0-rmse:8.01983
[499]	validation_0-rmse:8.01845
XGB Regressor MAE: $ 7.05


In [8]:
# --- Combined output: signal + predicted price ---
results = X_test.copy()
results["actual_next_close"]    = y_prc_test.values
results["predicted_next_close"] = price_pred
results["actual_signal"]        = y_sig_test.values
results["predicted_signal"]     = clf_pred
print(results[["actual_next_close", "predicted_next_close", "actual_signal", "predicted_signal"]].tail(10))

     actual_next_close  predicted_next_close  actual_signal  predicted_signal
114         264.720001            258.734863              1                 1
115         263.750000            258.397736              0                 1
116         262.519989            257.186310              0                 1
117         260.290009            250.119888              0                 1
118         257.459991            254.420120              0                 1
119         259.880005            260.314606              1                 1
120         260.829987            256.938019              1                 1
121         260.809998            264.782532              0                 1
122         255.759995            269.472626              0                 0
123         250.119995            254.967743              0                 0


In [9]:
# --- Latest signal + next day price forecast ---
latest_scaled = X_test_scaled[[-1]]

prob            = xgb_clf.predict_proba(latest_scaled)[:, 1][0]
predicted_price = xgb_reg.predict(latest_scaled)[0]

if prob > 0.6:
    signal = "BUY"
elif prob < 0.4:
    signal = "SELL"
else:
    signal = "HOLD"

print(f"Signal:               {signal}")
print(f"Confidence:           {round(prob, 2)}")
print(f"Predicted Next Close: ${round(predicted_price, 2)}")

Signal:               SELL
Confidence:           0.30000001192092896
Predicted Next Close: $254.97000122070312
